## Hierarchical bootstrap confidence intervals

The per-layer profile of chapter 02 at the paper's full scale, 824 utterances × 41 integer SNRs × 18 DEMAND noises, with the confidence bands the paper draws around it (Sec. "Linearization of CKA Profiles", Fig. 3). Every interval in the paper is a **hierarchical cluster bootstrap** over the sampling units of the probing design, utterances and noise types, not a resample of the 41 points of the mean curve. The first part reproduces the published figure from the shipped tables; the second shows why the unit matters and cross-checks two independent implementations of the utterance-cluster bootstrap on the demo data; the third re-runs the published draw mechanism when the full parquet is available.

Runtime: under 60 seconds on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _rel(p):
    """Repo-relative path for display (absolute only if outside the repo)."""
    try:
        return p.relative_to(REPO)
    except ValueError:
        return p

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import numpy as np
import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {_rel(DATA_DIR)}")
print(f"Paper tables from: {_rel(TABLES)}")

## The published profile with its hierarchical intervals

`fits_muse_snr_jobB.csv` carries the OLS point estimates of the mean curve; `muse_profile_values_18_cluster.csv` carries the same estimates with their 95% percentile intervals from a utterance × noise bootstrap (1,000 replicates, seed 0). The two files must agree on the point estimates to machine precision.

In [ ]:
from se_probe.layers import layout, probed_layers, short_labels

fits = pd.read_csv(TABLES / "snr/fits_muse_snr_jobB.csv")
cis = pd.read_csv(TABLES / "snr/muse_profile_values_18_cluster.csv")
assert list(fits["layer"]) == probed_layers("muse", fits["layer"]) == list(cis["layer"]), "layer order is not architectural depth"
assert np.allclose(fits["alpha"], cis["alpha"], atol=1e-12) and np.allclose(fits["beta"], cis["beta"], atol=1e-12)

prof = fits.merge(cis.drop(columns=["alpha", "beta", "r2"]), on="layer")
prof["label"] = short_labels("muse", prof["layer"])
peak = prof.loc[prof["beta"].idxmax(), "label"]
print(f"intercept range: {prof['alpha'].max():.3f} (first encoder layer) -> {prof['alpha'].min():.3f} (first refinement layer)   (paper: 0.99 -> 0.25; the paper truncates 0.997 to 0.99)")
print(f"peak sensitivity layer: {peak}   (paper: Dec-L2.0)")
print(f"min R^2 over the 24 fits: {prof['r2'].min():.4f}   (paper: >= 0.96)")
assert peak == "Dec-L2.0" and prof["r2"].min() >= 0.96
assert prof["alpha"].max() > 0.99 and abs(prof["alpha"].min() - 0.25) < 0.01  # the paper rounds to two decimals
prof[["label", "alpha", "alpha_ci_lo", "alpha_ci_hi", "beta", "beta_ci_lo", "beta_ci_hi", "r2"]].round(4).head(8)

In [ ]:
import matplotlib.pyplot as plt

from se_probe.plotting import profile_panel_pair

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
profile_panel_pair(ax1, ax2, prof, layout("muse"), axis_tag="SNR",
                   beta_ci=("beta_ci_lo", "beta_ci_hi"), alpha_ci=("alpha_ci_lo", "alpha_ci_hi"))
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("MUSE under additive noise, 824 x 41 x 18, bands = 95% hierarchical bootstrap CI", y=1.06, fontsize=10)
plt.tight_layout();

## Why the sampling unit, not the curve, must be resampled

A row of the results table is one (utterance, noise, SNR, layer) measurement. Utterances are crossed with every noise type and nested in speakers, so rows are far from independent. Resampling the 41 points of the mean curve only propagates the residual scatter about the fitted line, which is negligible when R² is above 0.96, and silently drops the dominant variance components (utterance-to-utterance and noise-to-noise). The paper therefore resamples whole utterances and whole noise types.

The demo parquet holds one noise type (TBUS) at five SNRs for about fifty utterances, so a two-way bootstrap is not possible here; the utterance-cluster bootstrap is. Two implementations ship in the package: `se_probe.profiles.utterance_bootstrap` (resamples a `layers × utterances × levels` cube) and the general `se_probe.bootstrap.hierarchical_bootstrap` (weight-based cluster resampling of any statistic). Their random streams differ, so the agreement to check is the interval **width**, up to Monte-Carlo error.

In [ ]:
from se_probe.bootstrap import hierarchical_bootstrap, ols_alpha_beta
from se_probe.profiles import build_cube, utterance_bootstrap

muse_path = DATA_DIR / ("snr/cka_snr_muse.parquet" if (DATA_DIR / "snr").exists() else "cka_snr_muse_demo.parquet")
demo = pd.read_parquet(muse_path, columns=["clean_idx", "snr", "layer", "CKA", "noise_name"])
demo = demo[demo["noise_name"] == "TBUS"]
layers = probed_layers("muse", demo["layer"].unique())
sub = demo[demo["layer"].isin(layers)].reset_index(drop=True)
print(f"{len(sub):,} rows | {sub['clean_idx'].nunique()} utterances | SNRs {[float(s) for s in sorted(sub['snr'].unique())]} | {len(layers)} layers")

# implementation 1: cube resampling
cube, units, levels = build_cube(sub, layers, level_col="snr", unit_col="clean_idx")
ub = utterance_bootstrap(cube, levels, n_boot=300, seed=0, stats_keys=("alpha", "beta"))
width_cube = ub["beta_hi"] - ub["beta_lo"]

# implementation 2: weighted statistic under the general hierarchical bootstrap
layer_code = sub["layer"].map({L: i for i, L in enumerate(layers)}).to_numpy()
snr_vals = np.sort(sub["snr"].unique()).astype(float)
snr_code = sub["snr"].astype(float).map({s: i for i, s in enumerate(snr_vals)}).to_numpy()
key = layer_code * len(snr_vals) + snr_code
cka_vals = sub["CKA"].to_numpy()
K = len(layers) * len(snr_vals)

def beta_statistic(indices, weights):
    num = np.bincount(key[indices], weights=weights * cka_vals[indices], minlength=K)
    den = np.bincount(key[indices], weights=weights, minlength=K)
    curves = (num / den).reshape(len(layers), len(snr_vals))
    _, beta = ols_alpha_beta(curves, snr_vals)
    return beta

hb = hierarchical_bootstrap(beta_statistic, {"utterance": sub["clean_idx"].to_numpy()},
                            unit="utterance", n_boot=300, seed=0)
width_hb = hb.ci_high - hb.ci_low
print(hb.describe())
assert np.allclose(hb.estimate, ub["beta"], atol=1e-10), "point estimates differ between implementations"
ratio = np.median(width_hb / width_cube)
print(f"median width ratio (general / cube implementation) = {ratio:.3f}   (1.0 up to Monte-Carlo error)")
assert 0.75 < ratio < 1.33
cmp = pd.DataFrame({"label": short_labels("muse", layers), "beta": ub["beta"],
                    "width_cube": width_cube, "width_hierarchical": width_hb}).round(5)
cmp.head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
x = np.arange(len(layers))
ax.fill_between(x, ub["beta_lo"], ub["beta_hi"], color=MODEL_COLORS["muse"], alpha=0.2, label="utterance bootstrap (cube)")
ax.plot(x, hb.ci_low, ":", color="k", lw=1, label="hierarchical_bootstrap, unit=utterance")
ax.plot(x, hb.ci_high, ":", color="k", lw=1)
ax.plot(x, ub["beta"], "o-", color=MODEL_COLORS["muse"], ms=3, label="point estimate")
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title(f"Demo data ({sub['clean_idx'].nunique()} utterances, TBUS): two utterance-cluster bootstraps agree")
lay = layout("muse")
ax.set_xticks(lay["ticks"])
ax.set_xticklabels(lay["ticklabels"], rotation=30, ha="right")
for s in lay["seps"]:
    ax.axvline(s, color="k", ls="--", lw=0.5)
ax.legend(loc="best", fontsize=8)
plt.tight_layout();

## Re-running the published draw mechanism

The intervals in `muse_profile_values_18_cluster.csv` come from one `numpy.random.default_rng(0)` threaded through every model and every **hooked** layer (72 for MUSE) in sorted order, drawing utterances with `rng.choice` before noise types. `two_way_cluster_bootstrap(seed=0)` uses the same draw mechanism, but an exact, bit-identical replay needs the full 824-utterance table with every hooked layer passed in sorted order, which `scripts/analyze_snr_profiles.py --bootstrap` now does. The cell below passes only a few probed layers, so its random stream diverges from the published one after the first layer: the intervals it returns are statistically equivalent to the shipped ones (same resampling design, same number of replicates), not bit-identical, and the printed discrepancy is Monte-Carlo error rather than a bug. On a clone without `results_df/` the cell only reports that the branch is skipped.

In [ ]:
from se_probe.profiles import two_way_cluster_bootstrap

full = REPO / "results_df/snr/cka_snr_muse.parquet"
if full.exists():
    df_full = pd.read_parquet(full, columns=["clean_idx", "snr", "layer", "CKA", "noise_name"])
    n_noise = df_full["noise_name"].nunique()
    some = sorted(layers)[:3]  # same draw mechanism; a bit-identical replay needs all 72 hooked layers in sorted() order
    rep = two_way_cluster_bootstrap(df_full, some, level_col="snr", cluster_col="noise_name", n_boot=1000, seed=0)
    ref = cis.set_index("layer").loc[rep["layer"]]
    print(f"{n_noise} noise types; max |delta beta CI| vs shipped (Monte-Carlo error, not an exact replay): "
          f"{np.abs(rep[['beta_ci_lo', 'beta_ci_hi']].to_numpy() - ref[['beta_ci_lo', 'beta_ci_hi']].to_numpy()).max():.2e}")
    rep
else:
    print("results_df/snr/cka_snr_muse.parquet not present: full-table re-run skipped (python scripts/setup.py --full-data fetches it).")